Group 162 Members (Contribution is 100% by each member)
1. 2025ae05368 - Sanjay S Betageri
2. 2025ae05921 - WUDALI LAKSHMI NARSU DHANISTA
3. 2025ae05786 - Simran Ahuja
4. 2025ae05885 - Sumit Sharma

# Assignment 2B: RAG Pipeline

## Overview
Build and evaluate a compact retrieval-augmented generation pipeline over the AWS documentation corpus prepared for Assignment 1B. The work proceeds from dense retrieval and chunk-size selection (Part A), through BM25 and hybrid retrieval (Part B), reranking (Part C), and grounded answer generation plus LLM judging (Part D). The notebook is the single executable report; saved intermediate data and results live under this folder.

| Part | Deliverable | Marks |
|---|---|---:|
| A | Dense search, chunk-size comparison, and selected chunk size | 5 |
| B | Dense, BM25, and RRF hybrid comparison | 4 |
| C | Cross-encoder reranking and ranking comparison | 4 |
| D | Grounded answers, LLM-judge results, spot-check, and failure analysis | 7 |

## Reuse from Assignment 1B
- The six cleaned AWS text files are copied into this project's `domain_corpus/` folder; no external corpus path is needed at runtime.
- The ten selected C3 prompts and the local evaluation manifest are in `data/c3_prompts.json` and `data/evaluation_queries.json`. The notebook verifies that they match.
- Assignment 1B's original evaluation file contains five prompts, while C3 provides ten prompts without gold answers. Curate a concise expected answer and exact corpus answer span for each C3 prompt before measuring hit rates; do not treat generated model outputs as gold references.
- The Assignment 1B generator is `mistralai/Mistral-7B-v0.1`; its QLoRA adapter is copied into `adapters/mistral7b-aws-qlora/`. Test grounding behavior before generating all answers because earlier outputs showed repetition and unsupported recommendations.
- The corpus, prompts, adapter, dependencies, and results are kept in this folder. The original Assignment 1B project remains unchanged.

Setup Requirements

In [1]:
# Install only missing or incompatible requirements into this notebook's active virtual environment.
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from packaging.requirements import Requirement
import subprocess
import sys

if sys.prefix == sys.base_prefix:
    raise RuntimeError(
        f'Global Python is selected ({sys.executable}). Select this project\'s .venv312 kernel before installing requirements.'
    )

requirements_file = Path.cwd().resolve() / 'requirements.txt'
if not requirements_file.is_file():
    raise FileNotFoundError(f'requirements.txt not found in {Path.cwd()}')

requirements = [
    Requirement(line.strip())
    for line in requirements_file.read_text(encoding='utf-8').splitlines()
    if line.strip() and not line.lstrip().startswith('#')
]
requirements = [
    requirement for requirement in requirements
    if requirement.marker is None or requirement.marker.evaluate()
]

def find_missing_requirements():
    missing = []
    for requirement in requirements:
        try:
            installed_version = version(requirement.name)
        except PackageNotFoundError:
            missing.append(str(requirement))
            continue
        if requirement.specifier and not requirement.specifier.contains(installed_version, prereleases=True):
            missing.append(str(requirement))
    return missing


missing_requirements = find_missing_requirements()
print(f'Active virtual environment: {sys.prefix}')
if missing_requirements:
    print(f'Installing {len(missing_requirements)} missing or incompatible requirements: {missing_requirements}')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)],
        check=True,
    )
    remaining_requirements = find_missing_requirements()
    if remaining_requirements:
        raise RuntimeError(f'Requirements still missing or incompatible: {remaining_requirements}')
else:
    print(f'All {len(requirements)} requirements are already satisfied; skipping pip install.')

Active virtual environment: d:\Docs\Edu\3rd-sem\LLM-GenAI\2b-assignment-RAG\.venv312
All 13 requirements are already satisfied; skipping pip install.


## Part A: Dense Search and Chunk Size

### A0. Evaluation setup
Use the same ten C3 prompts throughout Parts A-C. Before calculating hit rates, fill `expected_answer` and `answer_span` in `data/evaluation_queries.json` for every query. `answer_span` must be a short, exact passage from one of the cleaned corpus files; a retrieved chunk is a hit when it contains that passage (ignoring case and repeated whitespace). Keep the expected answer grounded in the same source passage.

### A1. Dense retrieval
Use `BAAI/bge-small-en-v1.5` with its own tokenizer and a FAISS `IndexFlatIP` index. Embeddings and query vectors are L2-normalized, so inner product is cosine similarity. The notebook records the model's actual maximum sequence length, vector dimension, embedding batch size, and combined embedding/index build time. Chunk tokenization uses the embedding model tokenizer.

### A2. Chunk-size comparison
Compare fixed, non-overlapping 100-token and 400-token chunks. The assignment checklist specifies 100 and 400; one extracted table says 500, so this notebook follows the checklist. The selected embedding model must accommodate 400 tokens plus special tokens; the code checks this rather than silently accepting truncation. For each size, report corpus chunk count, hit rate@3 over the ten queries, and average number of chunk tokens returned in the top three. Select the Part B-D chunk size only after reviewing those results and the generator's context budget.

### Part A execution order
1. Verify the input paths and load the ten C3 prompts and six cleaned corpus files.
2. Curate the ten expected answers and answer spans in the query manifest.
3. Create token-exact chunks at 100 and 400 tokens, build dense indexes, and evaluate top-three hits.
4. Record the measured settings and choose a chunk size for Parts B-D.

In [2]:
import json
from pathlib import Path

PROJECT_DIR = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / 'data' / 'evaluation_queries.json').is_file()
     and (path / 'data' / 'c3_prompts.json').is_file()
     and (path / 'domain_corpus').is_dir()
     and (path / 'adapters' / 'mistral7b-aws-qlora' / 'adapter_config.json').is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        'Could not locate the standalone RAG project (data/, domain_corpus/, adapters/).'
    )

A2B_DIR = PROJECT_DIR
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
CORPUS_DIR = PROJECT_DIR / 'domain_corpus'
ADAPTER_PATH = PROJECT_DIR / 'adapters' / 'mistral7b-aws-qlora'
C3_PROMPTS_PATH = DATA_DIR / 'c3_prompts.json'
QUERY_MANIFEST_PATH = DATA_DIR / 'evaluation_queries.json'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

queries = json.loads(QUERY_MANIFEST_PATH.read_text(encoding='utf-8'))
c3_prompts = json.loads(C3_PROMPTS_PATH.read_text(encoding='utf-8'))
corpus_paths = sorted(CORPUS_DIR.glob('*.txt'))

assert len(queries) == 10, f'Expected 10 queries, found {len(queries)}.'
assert len(c3_prompts) == 10, f'Expected 10 C3 prompts, found {len(c3_prompts)}.'
assert [(q['id'], q['prompt']) for q in queries] == [(q['id'], q['prompt']) for q in c3_prompts], 'Local query manifest differs from the copied C3 prompt set.'
assert len(corpus_paths) == 6, f'Expected six copied corpus files, found {len(corpus_paths)}.'

print(f'RAG project root: {PROJECT_DIR}')
print(f'Queries: {len(queries)} (local C3 prompt set verified)')
print(f'Cleaned corpus documents: {len(corpus_paths)}')
print(f'Generator adapter: {ADAPTER_PATH}')
print('Reference answers/spans curated:', sum(bool(q.get('expected_answer') and q.get('answer_span')) for q in queries), '/ 10')

RAG project root: D:\Docs\Edu\3rd-sem\LLM-GenAI\2b-assignment-RAG
Queries: 10 (local C3 prompt set verified)
Cleaned corpus documents: 6
Generator adapter: D:\Docs\Edu\3rd-sem\LLM-GenAI\2b-assignment-RAG\adapters\mistral7b-aws-qlora
Reference answers/spans curated: 10 / 10


### A0.1 Curate gold references
The prompt set is fixed, but C3 did not save expected answers or answer spans. Fill the two null fields per query in `data/evaluation_queries.json` by reading the corpus. Keep the span short enough to identify the relevant fact and exact enough to test whether a retrieved chunk contains it. This manual reference step is required for defensible hit-rate calculations.

In [3]:
from collections import Counter

corpus_documents = {path.stem: path.read_text(encoding='utf-8') for path in corpus_paths}

def tokenize_chunks(text, tokenizer, chunk_size, source, first_chunk_id=0):
    """Split text into fixed token windows while retaining exact source substrings."""
    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True, truncation=False)
    offsets = encoded['offset_mapping']
    chunks = []
    for chunk_number, token_start in enumerate(range(0, len(offsets), chunk_size), start=first_chunk_id):
        token_end = min(token_start + chunk_size, len(offsets))
        char_start = offsets[token_start][0]
        char_end = offsets[token_end - 1][1]
        chunk_text = text[char_start:char_end].strip()
        if chunk_text:
            chunks.append({
                'id': f'{source}:{chunk_number}',
                'source': source,
                'text': chunk_text,
                'token_count': token_end - token_start,
            })
    return chunks

def build_chunks(documents, tokenizer, chunk_size):
    chunks = []
    for source, text in documents.items():
        chunks.extend(tokenize_chunks(text, tokenizer, chunk_size, source))
    return chunks

def normalize_text(text):
    return ' '.join(text.lower().split())

def answer_span_is_present(answer_span, chunk_text):
    return normalize_text(answer_span) in normalize_text(chunk_text)

### A1. Build dense retrieval
The complete Assignment 2B dependency set is in `requirements.txt`; it covers dense embeddings, FAISS, BM25, cross-encoder reranking, Mistral adapter inference, and HTML export. Install it in a separate Prayogshala environment/kernel so package changes do not affect Assignment 1B. The manifest pins PyTorch 2.3.1; confirm that this matches the lab image's CUDA runtime and use the lab-approved PyTorch wheel if needed. Do not install this stack into the Assignment 1B environment. Embeddings are normalized and searched with inner product; this is equivalent to cosine similarity for normalized vectors. The 400-token chunk size is checked against the loaded model's actual limit, including special tokens.

In [4]:
import time
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_ID = 'BAAI/bge-small-en-v1.5'
EMBEDDING_BATCH_SIZE = 32
EMBEDDING_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SentenceTransformer(EMBEDDING_MODEL_ID, device=EMBEDDING_DEVICE)
embedding_tokenizer = embedder.tokenizer
embedding_max_length = int(embedder.max_seq_length)
special_token_count = embedding_tokenizer.num_special_tokens_to_add(pair=False)

assert 400 + special_token_count <= embedding_max_length, (
    f'400-token chunks plus {special_token_count} special tokens exceed the model limit '
    f'of {embedding_max_length}; choose a longer-context model or revise the comparison sizes.'
)

def validate_chunks_fit_model(chunks, model=embedder, batch_size=EMBEDDING_BATCH_SIZE):
    """Reject chunks that the model tokenizer would truncate, including special tokens."""
    max_length = int(model.max_seq_length)
    longest_input_length = 0
    for start in range(0, len(chunks), batch_size):
        chunk_batch = chunks[start:start + batch_size]
        encoded = model.tokenizer(
            [chunk['text'] for chunk in chunk_batch],
            add_special_tokens=True,
            padding=False,
            truncation=False,
        )
        for chunk, input_ids in zip(chunk_batch, encoded['input_ids']):
            input_length = len(input_ids)
            longest_input_length = max(longest_input_length, input_length)
            if input_length > max_length:
                raise ValueError(
                    f"Chunk {chunk['id']} encodes to {input_length} tokens including "
                    f"special tokens, exceeding the model limit of {max_length}."
                )
    return longest_input_length

def build_dense_index(chunks, model=embedder, batch_size=EMBEDDING_BATCH_SIZE):
    """Embed chunks, build a normalized-vector FAISS IndexFlatIP, and report build time."""
    if not chunks:
        raise ValueError('Cannot build an index with no chunks.')
    longest_input_length = validate_chunks_fit_model(chunks, model, batch_size)
    started = time.perf_counter()
    vectors = model.encode(
        [chunk['text'] for chunk in chunks],
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype('float32')
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    elapsed = time.perf_counter() - started
    return index, vectors.shape[1], elapsed, longest_input_length

def search_dense(query, index, chunks, model=embedder, top_k=5):
    query_vector = model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    ).astype('float32')
    scores, indices = index.search(query_vector, min(top_k, len(chunks)))
    return [(chunks[int(i)], float(score)) for i, score in zip(indices[0], scores[0]) if i >= 0]

d:\Docs\Edu\3rd-sem\LLM-GenAI\2b-assignment-RAG\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### A2. Evaluate 100-token and 400-token chunks
The evaluation cell builds a separate dense index for each size. Hit rate@3 is computed only when all ten answer spans are present; otherwise it stops with a clear message instead of reporting a misleading metric. Tokens sent per query is the sum of the stored token counts for the top three retrieved chunks, averaged over queries.

In [5]:
def evaluate_chunk_set(chunks, index, query_rows, top_k=3):
    if not all(row.get('answer_span') for row in query_rows):
        raise ValueError('Curate an answer_span for all ten queries before computing hit rate.')
    hits = []
    tokens_per_query = []
    for row in query_rows:
        retrieved = search_dense(row['prompt'], index, chunks, top_k=top_k)
        hits.append(any(answer_span_is_present(row['answer_span'], chunk['text']) for chunk, _ in retrieved))
        tokens_per_query.append(sum(chunk['token_count'] for chunk, _ in retrieved))
    return {
        'total_chunks': len(chunks),
        'hit_rate_at_3_pct': 100 * sum(hits) / len(hits),
        'avg_top3_tokens_per_query': sum(tokens_per_query) / len(tokens_per_query),
    }

gold_references_ready = all(row.get('expected_answer') and row.get('answer_span') for row in queries)
if not gold_references_ready:
    print('Part A retrieval functions are ready, but metrics are blocked until all ten expected answers and answer spans are curated in data/evaluation_queries.json.')
else:
    part_a_results = {}
    for chunk_size in (100, 400):
        chunks = build_chunks(corpus_documents, embedding_tokenizer, chunk_size)
        index, vector_size, build_seconds, max_chunk_input_tokens = build_dense_index(chunks)
        metrics = evaluate_chunk_set(chunks, index, queries)
        part_a_results[chunk_size] = {
            **metrics,
            'embedding_model': EMBEDDING_MODEL_ID,
            'vector_size': vector_size,
            'similarity': 'cosine similarity (L2-normalized vectors, IndexFlatIP)',
            'batch_size': EMBEDDING_BATCH_SIZE,
            'embedding_max_length': embedding_max_length,
            'max_chunk_input_tokens_including_special_tokens': max_chunk_input_tokens,
            'index_build_seconds': build_seconds,
        }
        print(f'{chunk_size}-token chunks: {part_a_results[chunk_size]}')
    (RESULTS_DIR / 'part_a_chunk_comparison.json').write_text(
        json.dumps(part_a_results, indent=2), encoding='utf-8'
    )
    print(f"Saved Part A metrics to {RESULTS_DIR / 'part_a_chunk_comparison.json'}")

Token indices sequence length is longer than the specified maximum sequence length for this model (29920 > 512). Running this sequence through the model will result in indexing errors
Batches: 100%|██████████| 58/58 [01:49<00:00,  1.89s/it]


100-token chunks: {'total_chunks': 1837, 'hit_rate_at_3_pct': 0.0, 'avg_top3_tokens_per_query': 300.0, 'embedding_model': 'BAAI/bge-small-en-v1.5', 'vector_size': 384, 'similarity': 'cosine similarity (L2-normalized vectors, IndexFlatIP)', 'batch_size': 32, 'embedding_max_length': 512, 'max_chunk_input_tokens_including_special_tokens': 103, 'index_build_seconds': 109.92496129999927}


Batches: 100%|██████████| 15/15 [02:35<00:00, 10.38s/it]


400-token chunks: {'total_chunks': 461, 'hit_rate_at_3_pct': 20.0, 'avg_top3_tokens_per_query': 1200.0, 'embedding_model': 'BAAI/bge-small-en-v1.5', 'vector_size': 384, 'similarity': 'cosine similarity (L2-normalized vectors, IndexFlatIP)', 'batch_size': 32, 'embedding_max_length': 512, 'max_chunk_input_tokens_including_special_tokens': 403, 'index_build_seconds': 155.6971782999899}
Saved Part A metrics to D:\Docs\Edu\3rd-sem\LLM-GenAI\2b-assignment-RAG\results\part_a_chunk_comparison.json


### Part A decision record
The 400-token chunks produced a higher hit rate@3 on this ten-query set, while sending four times as many tokens in the top three. The copied Assignment 1B Mistral-7B-v0.1 model has ample context for this 1,200-token retrieved context, so 400 tokens is selected for Parts B-D; retrieval quality should still be reassessed on the larger evaluation in later parts.

| Chunk size | Total chunks | Hit rate@3 (%) | Avg. top-3 tokens/query |
|---:|---:|---:|---:|
| 100 | 1,837 | 0 | 300 |
| 400 | 461 | 20 | 1,200 |

**Selected size and rationale:** 400 tokens. It retrieved an answer span for 2/10 queries versus 0/10 with 100-token chunks. Its 1,200-token top-three context is still modest relative to the generator's context window, so the observed retrieval gain is worth the extra context.

## Part B: Keyword and Hybrid Search

Use the selected 400-token chunks from Part A and the same ten query prompts and answer spans. Dense retrieval and BM25 each return their top five for comparison; RRF fuses each method's top 20, with ranks starting at 1 and `k=60`, then returns five. Average query latency excludes one warm-up run per method. Hit rates are based on the exact curated answer spans.

In [6]:
import re
from IPython.display import Markdown, display
from rank_bm25 import BM25Okapi

SELECTED_CHUNK_SIZE = 400
assert len(queries) == 10, f'Part B requires the same ten queries; found {len(queries)}.'
assert len(chunks) == part_a_results[SELECTED_CHUNK_SIZE]['total_chunks'], (
    'Part B chunks do not match the selected Part A chunk set.'
)
assert all(row.get('answer_span') for row in queries), 'All ten answer spans must be curated.'


def keyword_tokens(text):
    """Keep AWS identifiers such as s3:GetObject and ARN segments searchable."""
    return re.findall(r'[a-z0-9]+(?:[._:/-][a-z0-9]+)*', text.lower())


bm25_corpus = [keyword_tokens(chunk['text']) for chunk in chunks]
bm25_index = BM25Okapi(bm25_corpus)


def search_bm25(query, index=bm25_index, chunk_set=chunks, top_k=5):
    scores = index.get_scores(keyword_tokens(query))
    positions = np.argsort(-scores, kind='stable')[:min(top_k, len(chunk_set))]
    return [(chunk_set[int(position)], float(scores[position])) for position in positions]


def reciprocal_rank_fusion(dense_results, keyword_results, top_k=5, rrf_constant=60):
    """Fuse the top 20 from each retriever; a missing rank contributes no score."""
    fused_scores = {}
    chunks_by_id = {}
    for ranked_results in (dense_results[:20], keyword_results[:20]):
        for rank, (chunk, _) in enumerate(ranked_results, start=1):
            chunk_id = chunk['id']
            chunks_by_id[chunk_id] = chunk
            fused_scores[chunk_id] = fused_scores.get(chunk_id, 0.0) + 1 / (rrf_constant + rank)
    ranked_ids = sorted(fused_scores, key=lambda chunk_id: (-fused_scores[chunk_id], chunk_id))
    return [(chunks_by_id[chunk_id], fused_scores[chunk_id]) for chunk_id in ranked_ids[:top_k]]


def search_hybrid(query, top_k=5):
    dense_top20 = search_dense(query, index, chunks, top_k=20)
    bm25_top20 = search_bm25(query, top_k=20)
    return reciprocal_rank_fusion(dense_top20, bm25_top20, top_k=top_k)


def benchmark_retriever(search_function, result_count):
    """Run one untimed warm-up, then time one top-k result set per evaluation query."""
    search_function(queries[0]['prompt'], result_count)
    measured = []
    for query_row in queries:
        started = time.perf_counter()
        results = search_function(query_row['prompt'], result_count)
        measured.append({
            'results': results,
            'latency_ms': (time.perf_counter() - started) * 1000,
        })
    return measured


dense_runs = benchmark_retriever(
    lambda query, top_k: search_dense(query, index, chunks, top_k=top_k), 5
)
bm25_runs = benchmark_retriever(
    lambda query, top_k: search_bm25(query, top_k=top_k), 5
)
hybrid_runs = benchmark_retriever(search_hybrid, 5)
retrieval_runs = {
    'Dense': dense_runs,
    'BM25': bm25_runs,
    'Hybrid (RRF)': hybrid_runs,
}


def retrieval_hits(ranked_results, answer_span):
    top_three = ranked_results[:3]
    return {
        'hit_at_1': bool(ranked_results) and answer_span_is_present(answer_span, ranked_results[0][0]['text']),
        'hit_at_3': any(answer_span_is_present(answer_span, chunk['text']) for chunk, _ in top_three),
    }


part_b_results = {}
per_query_results = {}
for method_name, runs in retrieval_runs.items():
    query_records = []
    for query_row, run in zip(queries, runs):
        hits = retrieval_hits(run['results'], query_row['answer_span'])
        query_records.append({
            'query_id': query_row['id'],
            'latency_ms': run['latency_ms'],
            'hit_at_1': hits['hit_at_1'],
            'hit_at_3': hits['hit_at_3'],
            'top5_chunk_ids': [chunk['id'] for chunk, _ in run['results']],
        })
    per_query_results[method_name] = query_records
    part_b_results[method_name] = {
        'average_query_latency_ms': sum(record['latency_ms'] for record in query_records) / len(query_records),
        'hit_rate_at_1_pct': 100 * sum(record['hit_at_1'] for record in query_records) / len(query_records),
        'hit_rate_at_3_pct': 100 * sum(record['hit_at_3'] for record in query_records) / len(query_records),
        'per_query': query_records,
    }

part_b_results['settings'] = {
    'chunk_size_tokens': SELECTED_CHUNK_SIZE,
    'queries': len(queries),
    'top_k_reported': 5,
    'rrf_fusion_depth_per_retriever': 20,
    'rrf_constant': 60,
    'warmup_runs_excluded_per_method': 1,
    'bm25_tokenizer': 'lowercase regex preserving AWS identifier punctuation',
}

method_names = list(retrieval_runs)
best_method = max(
    method_names,
    key=lambda name: (
        part_b_results[name]['hit_rate_at_3_pct'],
        part_b_results[name]['hit_rate_at_1_pct'],
        -part_b_results[name]['average_query_latency_ms'],
    ),
)
per_query_by_method = {
    name: {row['query_id']: row for row in rows}
    for name, rows in per_query_results.items()
}
example_id = next(
    (row['query_id'] for row in per_query_results[best_method]
     if row['hit_at_3'] and any(
         not per_query_by_method[other][row['query_id']]['hit_at_3']
         for other in method_names if other != best_method
     )),
    queries[0]['id'],
)
example_best = per_query_by_method[best_method][example_id]
example_lagging = [
    other for other in method_names
    if not per_query_by_method[other][example_id]['hit_at_3']
]
if example_best['hit_at_3'] and example_lagging:
    example_sentence = f"For {example_id}, {best_method} retrieved the answer span in its top three while {', '.join(example_lagging)} did not."
else:
    example_sentence = f"For {example_id}, {best_method} had a top-three hit rate of {part_b_results[best_method]['hit_rate_at_3_pct']:.0f}% overall."

markdown_table = [
    '| Method | Average query latency (ms) | Hit rate@1 (%) | Hit rate@3 (%) |',
    '|---|---:|---:|---:|',
]
for method_name in method_names:
    metrics = part_b_results[method_name]
    markdown_table.append(
        f"| {method_name} | {metrics['average_query_latency_ms']:.2f} | "
        f"{metrics['hit_rate_at_1_pct']:.0f} | {metrics['hit_rate_at_3_pct']:.0f} |"
    )
summary = (
    f"**Result:** {best_method} performed best by hit rate@3 "
    f"({part_b_results[best_method]['hit_rate_at_3_pct']:.0f}%), with hit rate@1 "
    f"{part_b_results[best_method]['hit_rate_at_1_pct']:.0f}% and mean latency "
    f"{part_b_results[best_method]['average_query_latency_ms']:.2f} ms. {example_sentence}"
)
display(Markdown('\n'.join(markdown_table) + '\n\n' + summary))

(PART_B_RESULTS_PATH := RESULTS_DIR / 'part_b_retrieval_comparison.json').write_text(
    json.dumps(part_b_results, indent=2), encoding='utf-8'
)
print(f'Saved Part B metrics and per-query retrieval evidence to {PART_B_RESULTS_PATH}')

| Method | Average query latency (ms) | Hit rate@1 (%) | Hit rate@3 (%) |
|---|---:|---:|---:|
| Dense | 80.67 | 0 | 20 |
| BM25 | 6.66 | 50 | 80 |
| Hybrid (RRF) | 62.48 | 10 | 40 |

**Result:** BM25 performed best by hit rate@3 (80%), with hit rate@1 50% and mean latency 6.66 ms. For c3_01_ec2_ssh, BM25 retrieved the answer span in its top three while Dense did not.

Saved Part B metrics and per-query retrieval evidence to D:\Docs\Edu\3rd-sem\LLM-GenAI\2b-assignment-RAG\results\part_b_retrieval_comparison.json


## Part C: Cross-Encoder Reranking

Part B selected BM25 as the current best method. This section chooses the best method from the measured Part B results, retrieves ten candidates per query, scores query-candidate pairs with `cross-encoder/ms-marco-MiniLM-L-6-v2`, and retains the top three.

### What the code does
1. Select the Part B method with the strongest hit@3, breaking ties by hit@1 and then latency.
2. Retrieve ten chunks for each of the same ten questions.
3. Score each question together with each candidate passage using a cross-encoder. The code checks pair token lengths before scoring so the reranker does not silently truncate candidates.
4. Display the top three chunk IDs and short passage previews before and after reranking for every query.
5. Report mean cross-encoder scoring time per query (after one warm-up), how often the first result changed, and hit@1 before and after. Hit@1 before uses the Part B measurement; hit@1 after checks the curated exact answer spans.

The next code cell downloads/loads the cross-encoder model if it is not cached. Review that cell before running it.

In [7]:
import time
import numpy as np
import torch
from IPython.display import Markdown, display
from sentence_transformers import CrossEncoder

RERANKER_MODEL_ID = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
RERANKER_MAX_LENGTH = 512
RERANKER_BATCH_SIZE = 32
CANDIDATE_COUNT = 10
FINAL_RESULT_COUNT = 3

available_methods = ('Dense', 'BM25', 'Hybrid (RRF)')
best_part_b_method = max(
    (name for name in available_methods if name in part_b_results),
    key=lambda name: (
        part_b_results[name]['hit_rate_at_3_pct'],
        part_b_results[name]['hit_rate_at_1_pct'],
        -part_b_results[name]['average_query_latency_ms'],
    ),
)
assert len(queries) == 10, f'Part C requires the same ten queries; found {len(queries)}.'
assert all(row.get('answer_span') for row in queries), 'All ten answer spans must be curated.'

candidate_searchers = {
    'Dense': lambda query, top_k: search_dense(query, index, chunks, top_k=top_k),
    'BM25': lambda query, top_k: search_bm25(query, top_k=top_k),
    'Hybrid (RRF)': lambda query, top_k: search_hybrid(query, top_k=top_k),
}
candidate_sets = {
    row['id']: candidate_searchers[best_part_b_method](row['prompt'], CANDIDATE_COUNT)
    for row in queries
}
assert all(len(candidates) == CANDIDATE_COUNT for candidates in candidate_sets.values()), (
    f'Expected {CANDIDATE_COUNT} candidates for every query.'
)

reranker_device = 'cuda' if torch.cuda.is_available() else 'cpu'
reranker = CrossEncoder(
    RERANKER_MODEL_ID,
    device=reranker_device,
    max_length=RERANKER_MAX_LENGTH,
)

for query_row in queries:
    candidates = candidate_sets[query_row['id']]
    encoded_pairs = reranker.tokenizer(
        [query_row['prompt']] * len(candidates),
        [chunk['text'] for chunk, _ in candidates],
        add_special_tokens=True,
        padding=False,
        truncation=False,
    )
    longest_pair_length = max(len(input_ids) for input_ids in encoded_pairs['input_ids'])
    if longest_pair_length > RERANKER_MAX_LENGTH:
        raise ValueError(
            f"{query_row['id']} has a query-passage pair of {longest_pair_length} tokens, "
            f'exceeding the reranker limit of {RERANKER_MAX_LENGTH}; no pair will be truncated.'
        )

warmup_query = queries[0]
warmup_pairs = [
    (warmup_query['prompt'], chunk['text'])
    for chunk, _ in candidate_sets[warmup_query['id']]
]
reranker.predict(
    warmup_pairs,
    batch_size=RERANKER_BATCH_SIZE,
    show_progress_bar=False,
)

part_c_per_query = []
for query_row in queries:
    candidates = candidate_sets[query_row['id']]
    pairs = [(query_row['prompt'], chunk['text']) for chunk, _ in candidates]

    rerank_started = time.perf_counter()
    scores = np.asarray(
        reranker.predict(
            pairs,
            batch_size=RERANKER_BATCH_SIZE,
            show_progress_bar=False,
        ),
        dtype='float32',
    ).reshape(-1)
    rerank_latency_ms = (time.perf_counter() - rerank_started) * 1000

    reranked_positions = np.argsort(-scores, kind='stable')[:FINAL_RESULT_COUNT]
    before_top3 = candidates[:FINAL_RESULT_COUNT]
    after_top3 = [
        (candidates[int(position)][0], float(scores[int(position)]))
        for position in reranked_positions
    ]
    before_first_id = before_top3[0][0]['id']
    after_first_id = after_top3[0][0]['id']
    before_part_b_row = next(
        row for row in per_query_results[best_part_b_method]
        if row['query_id'] == query_row['id']
    )
    after_hit_at_1 = answer_span_is_present(query_row['answer_span'], after_top3[0][0]['text'])

    def describe_ranked_chunks(ranked_chunks):
        descriptions = []
        for rank, (chunk, score) in enumerate(ranked_chunks, start=1):
            preview = ' '.join(chunk['text'].split())[:140].replace('|', '/')
            descriptions.append({
                'rank': rank,
                'chunk_id': chunk['id'],
                'source': chunk['source'],
                'score': float(score),
                'preview': preview,
            })
        return descriptions

    part_c_per_query.append({
        'query_id': query_row['id'],
        'reranking_latency_ms': rerank_latency_ms,
        'top1_changed': before_first_id != after_first_id,
        'hit_at_1_before': bool(before_part_b_row['hit_at_1']),
        'hit_at_1_after': bool(after_hit_at_1),
        'top3_before': describe_ranked_chunks(before_top3),
        'top3_after': describe_ranked_chunks(after_top3),
    })

reranking_time_per_query_ms = sum(
    row['reranking_latency_ms'] for row in part_c_per_query
) / len(part_c_per_query)
rank_change_rate_pct = 100 * sum(
    row['top1_changed'] for row in part_c_per_query
) / len(part_c_per_query)
hit_rate_at_1_before_pct = part_b_results[best_part_b_method]['hit_rate_at_1_pct']
hit_rate_at_1_after_pct = 100 * sum(
    row['hit_at_1_after'] for row in part_c_per_query
) / len(part_c_per_query)
hit_rate_change_pct = hit_rate_at_1_after_pct - hit_rate_at_1_before_pct

if hit_rate_change_pct > 0:
    reranking_conclusion = (
        f"Reranking helped on this query set: hit@1 increased from "
        f"{hit_rate_at_1_before_pct:.0f}% to {hit_rate_at_1_after_pct:.0f}% "
        f"({hit_rate_change_pct:+.0f} percentage points), at an average reranking cost "
        f"of {reranking_time_per_query_ms:.2f} ms per query."
    )
elif hit_rate_change_pct < 0:
    reranking_conclusion = (
        f"Reranking hurt hit@1 on this query set: it changed from "
        f"{hit_rate_at_1_before_pct:.0f}% to {hit_rate_at_1_after_pct:.0f}% "
        f"({hit_rate_change_pct:+.0f} percentage points), while adding "
        f"{reranking_time_per_query_ms:.2f} ms per query."
    )
else:
    reranking_conclusion = (
        f"Reranking did not change hit@1 on this query set ({hit_rate_at_1_before_pct:.0f}% "
        f"before and after), while adding {reranking_time_per_query_ms:.2f} ms per query."
    )

part_c_results = {
    'reranker_model': RERANKER_MODEL_ID,
    'reranker_max_length': RERANKER_MAX_LENGTH,
    'selected_part_b_method': best_part_b_method,
    'candidate_count': CANDIDATE_COUNT,
    'final_result_count': FINAL_RESULT_COUNT,
    'warmup_runs_excluded': 1,
    'reranking_time_per_query_ms': reranking_time_per_query_ms,
    'rank_change_rate_pct': rank_change_rate_pct,
    'hit_rate_at_1_before_pct': hit_rate_at_1_before_pct,
    'hit_rate_at_1_after_pct': hit_rate_at_1_after_pct,
    'per_query': part_c_per_query,
}

measurements_table = [
    '| Measure | Value |',
    '|---|---:|',
    f'| Reranking time per query (ms) | {reranking_time_per_query_ms:.2f} |',
    f'| Rank change rate (%) | {rank_change_rate_pct:.0f} |',
    f'| Hit rate@1 before reranking (%) | {hit_rate_at_1_before_pct:.0f} |',
    f'| Hit rate@1 after reranking (%) | {hit_rate_at_1_after_pct:.0f} |',
]
ranking_table = [
    '| Query | Top 3 before reranking | Top 3 after reranking | Top-1 changed | Hit@1 after |',
    '|---|---|---|---:|---:|',
]
for row in part_c_per_query:
    before = '<br>'.join(
        f"{item['rank']}. `{item['chunk_id']}` {item['preview']}"
        for item in row['top3_before']
    )
    after = '<br>'.join(
        f"{item['rank']}. `{item['chunk_id']}` {item['preview']}"
        for item in row['top3_after']
    )
    ranking_table.append(
        f"| {row['query_id']} | {before} | {after} | "
        f"{'Yes' if row['top1_changed'] else 'No'} | {'Yes' if row['hit_at_1_after'] else 'No'} |"
    )

display(Markdown('\n'.join(measurements_table) + '\n\n' + reranking_conclusion))
display(Markdown('\n'.join(ranking_table)))

PART_C_RESULTS_PATH = RESULTS_DIR / 'part_c_reranking_comparison.json'
PART_C_RESULTS_PATH.write_text(json.dumps(part_c_results, indent=2), encoding='utf-8')
print(f'Saved Part C rankings and metrics to {PART_C_RESULTS_PATH}')

d:\Docs\Edu\3rd-sem\LLM-GenAI\2b-assignment-RAG\.venv312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\z004na8z\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


| Measure | Value |
|---|---:|
| Reranking time per query (ms) | 1939.50 |
| Rank change rate (%) | 90 |
| Hit rate@1 before reranking (%) | 50 |
| Hit rate@1 after reranking (%) | 0 |

Reranking hurt hit@1 on this query set: it changed from 50% to 0% (-50 percentage points), while adding 1939.50 ms per query.

| Query | Top 3 before reranking | Top 3 after reranking | Top-1 changed | Hit@1 after |
|---|---|---|---:|---:|
| c3_01_ec2_ssh | 1. `ec2-ug:8` unsafe for production environments. In production, be sure to authorize access only from the appropriate individual IP address or range of a<br>2. `cfn-ug:51` Properties": { "BucketName": " amzn-s3-demo-bucket " } } } } YAML Resources: MyBucket: Type: AWS::S3::Bucket Properties: BucketName: amzn-s3<br>3. `ec2-ug:9` the operating system of the instance. If you can't connect to your instance, see Troubleshoot issues connecting to your Amazon EC2 Linux ins | 1. `cfn-ug:52` !Ref InstanceSecurityGroup - sg-12a4c434 KeyName: MyKey ImageId: ami-1234567890abcdef0 InstanceSecurityGroup: Type: AWS::EC2::SecurityGroup <br>2. `cfn-ug:51` Properties": { "BucketName": " amzn-s3-demo-bucket " } } } } YAML Resources: MyBucket: Type: AWS::S3::Bucket Properties: BucketName: amzn-s3<br>3. `ec2-ug:7` 't cover all possible options. To launch an instance 1. Open the Amazon EC2 console at https://console.aws.amazon.com/ec2/. 2. In the naviga | Yes | No |
| c3_02_ec2_console | 1. `vpc-ug:6` ﬁgure route tables Access the internet You control how the instances that you launch into a VPC access resources outside the VPC. A default <br>2. `vpc-ug:69` Example dual-stack VPC conﬁguration With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication between resou<br>3. `vpc-ug:10` will receive traﬃc only from other servers in the VPC, create a separate subnet for these servers in each Availability Zone. •If you have se | 1. `vpc-ug:6` ﬁgure route tables Access the internet You control how the instances that you launch into a VPC access resources outside the VPC. A default <br>2. `vpc-ug:13` an available IP address in the subnet range for you. For more information about network interfaces, see Elastic Network Interfaces in the Am<br>3. `vpc-ug:69` Example dual-stack VPC conﬁguration With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication between resou | No | No |
| c3_03_vpc_route | 1. `vpc-ug:10` will receive traﬃc only from other servers in the VPC, create a separate subnet for these servers in each Availability Zone. •If you have se<br>2. `vpc-ug:66` traﬃc, because NAT gateways do not support IPv6. To update the route table for a public subnet 1. Open the Amazon VPC console at https://con<br>3. `vpc-ug:69` Example dual-stack VPC conﬁguration With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication between resou | 1. `vpc-ug:66` traﬃc, because NAT gateways do not support IPv6. To update the route table for a public subnet 1. Open the Amazon VPC console at https://con<br>2. `vpc-ug:10` will receive traﬃc only from other servers in the VPC, create a separate subnet for these servers in each Availability Zone. •If you have se<br>3. `vpc-ug:6` ﬁgure route tables Access the internet You control how the instances that you launch into a VPC access resources outside the VPC. A default  | Yes | No |
| c3_04_s3_getobject | 1. `iam-ug:16` can use identity-based policies. Resource-based policies can grant cross-account access. To make a request in a diﬀerent account, a policy i<br>2. `lambda-dg:13` a trigger works, start by completing the Use an Amazon S3 trigger tutorial, or for a general overview of using triggers and instructions on <br>3. `iam-ug:0` User Guide AWS Identity and Access Management Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved. Require h | 1. `iam-ug:0` User Guide AWS Identity and Access Management Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved. Require h<br>2. `iam-ug:27` permissions to work with those services aren't included in the policy. Policies and IAM groups You can organize IAM users into IAM groups an<br>3. `iam-ug:57` about that entity. •aws iam get-service-last-accessed-details-with-entities c. Learn more about the identity-based policies that an identity | Yes | No |
| c3_05_s3_public_access | 1. `s3-userguide:58` the user can create a bucket. But the user does not own the bucket; the AWS account that the user belongs to owns the bucket. The user needs<br>2. `s3-userguide:25` bucket-owner-full-control canned ACL, you can add a bucket policy that allows only object uploads that use this ACL. •Object writer – The AW<br>3. `iam-ug:16` can use identity-based policies. Resource-based policies can grant cross-account access. To make a request in a diﬀerent account, a policy i | 1. `s3-userguide:25` bucket-owner-full-control canned ACL, you can add a bucket policy that allows only object uploads that use this ACL. •Object writer – The AW<br>2. `s3-userguide:58` the user can create a bucket. But the user does not own the bucket; the AWS account that the user belongs to owns the bucket. The user needs<br>3. `s3-userguide:11` ended user actions and application failures. For more information, see Retaining multiple versions of objects with S3 Versioning. Version ID | Yes | No |
| c3_06_lambda_timeout | 1. `lambda-dg:23` c01d1e2e1 REPORT RequestId: c3252230-c73d-49f6-8844-968c01d1e2e1 Duration: 933.59 ms Billed Duration: 934 ms Memory Size: 128 MB Max Memory <br>2. `lambda-dg:21` kes the function handler (the Invoke phase). Failures during the Restore phase If the Restore phase fails, Lambda emits error information in<br>3. `s3-userguide:4` Simple Notiﬁcation Service (Amazon SNS), Amazon Simple Queue Service (Amazon SQS), and AWS Lambda when a change is made to your S3 resources | 1. `lambda-dg:19` Function restarts from last checkpoint •Shutdown: Cleanup of durable state and resources Topics •Init phase •Failures during the Init phase <br>2. `lambda-dg:20` You might see gaps between your function's invocation and initialization phases. Depending on your function's runtime and memory conﬁguratio<br>3. `lambda-dg:17` Watch Logs. In addition to logging your function's output, the runtime also logs entries when function invocation starts and ends. This incl | Yes | No |
| c3_07_lambda_vpc | 1. `vpc-ug:6` ﬁgure route tables Access the internet You control how the instances that you launch into a VPC access resources outside the VPC. A default <br>2. `vpc-ug:69` Example dual-stack VPC conﬁguration With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication between resou<br>3. `vpc-ug:66` traﬃc, because NAT gateways do not support IPv6. To update the route table for a public subnet 1. Open the Amazon VPC console at https://con | 1. `vpc-ug:69` Example dual-stack VPC conﬁguration With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication between resou<br>2. `vpc-ug:10` will receive traﬃc only from other servers in the VPC, create a separate subnet for these servers in each Availability Zone. •If you have se<br>3. `vpc-ug:66` traﬃc, because NAT gateways do not support IPv6. To update the route table for a public subnet 1. Open the Amazon VPC console at https://con | Yes | No |
| c3_08_secretsmanager_iam | 1. `iam-ug:16` can use identity-based policies. Resource-based policies can grant cross-account access. To make a request in a diﬀerent account, a policy i<br>2. `lambda-dg:13` a trigger works, start by completing the Use an Amazon S3 trigger tutorial, or for a general overview of using triggers and instructions on <br>3. `lambda-dg:61` choose the AWS Region closest to your geographical location. aws s3api create-bucket --bucket amzn-s3-demo-bucket --region us-east-2 \ --cre | 1. `lambda-dg:13` a trigger works, start by completing the Use an Amazon S3 trigger tutorial, or for a general overview of using triggers and instructions on <br>2. `iam-ug:57` about that entity. •aws iam get-service-last-accessed-details-with-entities c. Learn more about the identity-based policies that an identity<br>3. `lambda-dg:14` . For example, if your function performs CRUD operations on a DynamoDB table, you can add the AmazonDynamoDBFullAccess policy to your role.  | Yes | No |
| c3_09_eventbridge_putevents | 1. `lambda-dg:39` potency •Use multiple AWS accounts for managing quotas Designing an application 47 Use services instead of custom code Serverless applicatio<br>2. `lambda-dg:14` . For example, if your function performs CRUD operations on a DynamoDB table, you can add the AmazonDynamoDBFullAccess policy to your role. <br>3. `lambda-dg:12` contains a runtime , a language-speciﬁc environment that relays event information and responses between Lambda and your function. Lambda pro | 1. `iam-ug:61` could look like the following example policy. JSON { "Version":"2012-10-17", "Statement": [ { "Sid": "FullAccessToSomeServices", "Effect": "<br>2. `iam-ug:0` User Guide AWS Identity and Access Management Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved. Require h<br>3. `iam-ug:54` the members of the user group and when they last accessed the service. •User – View the list of services that the user can access. You can a | Yes | No |
| c3_10_iam_role | 1. `iam-ug:38` in IAM Identity Center (either federated or created in the IAM Identity Center directory) use IAM roles to access resources. Before you star<br>2. `iam-ug:7` a speciﬁc type of resource-based policy for IAM roles. A role can have only one trust policy. •Identity-based policies (inline and managed) <br>3. `iam-ug:6` CLI to cryptographically sign requests. For more information, see AWS Signature Version 4 for API requests in the IAM User Guide . When you  | 1. `iam-ug:66` trust policies, role tags for ABAC, and conﬁgurable role paths. You can use it alongside permission sets or on its own. For more information<br>2. `iam-ug:7` a speciﬁc type of resource-based policy for IAM roles. A role can have only one trust policy. •Identity-based policies (inline and managed) <br>3. `iam-ug:21` create separate IAM users or users in IAM Identity Center for them. Instead, you can federate those user identities into AWS using either IA | Yes | No |

Saved Part C rankings and metrics to D:\Docs\Edu\3rd-sem\LLM-GenAI\2b-assignment-RAG\results\part_c_reranking_comparison.json


## Part D: Grounded Answer Generation

This step connects the top-three chunks retained by Part C to the local Mistral-7B-v0.1 QLoRA adapter. Each passage is numbered so the model can cite evidence as `[1]`, `[2]`, or `[3]`; it must answer `Not found in the documents` when the supplied passages do not support an answer. The code uses 4-bit loading on a CUDA-capable environment, generates one answer per evaluation query, and saves the answers and supporting chunk references to `results/part_d_generated_answers.json`.

This machine currently has no CUDA device and less than 9 GiB of available RAM, so this cell intentionally refuses CPU loading of the 7B model. Run it in the Assignment 2B virtual environment on a supported CUDA machine after reviewing the code.

In [ ]:
import json
import re
import torch
from IPython.display import Markdown, display
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError(
        'Mistral-7B 4-bit generation requires a CUDA-enabled environment. '
        'This machine has no CUDA device; switch to a supported GPU runtime before running this cell.'
    )

assert 'part_c_per_query' in globals() and len(part_c_per_query) == 10, (
    'Run the Part C reranking cell successfully before generating answers.'
)
assert ADAPTER_PATH.is_dir(), f'Local QLoRA adapter not found: {ADAPTER_PATH}'

adapter_config = json.loads((ADAPTER_PATH / 'adapter_config.json').read_text(encoding='utf-8'))
BASE_MODEL_ID = adapter_config['base_model_name_or_path']
GENERATION_MAX_NEW_TOKENS = 192
GENERATION_TEMPERATURE = 0.0

mistral_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, use_fast=False)
if mistral_tokenizer.pad_token_id is None:
    mistral_tokenizer.pad_token = mistral_tokenizer.eos_token
mistral_tokenizer.padding_side = 'left'

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map='auto',
)
mistral_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, is_trainable=False)
mistral_model.eval()
mistral_model.config.use_cache = True

mistral_context_limit = int(mistral_model.config.max_position_embeddings)
chunks_by_id = {chunk['id']: chunk for chunk in chunks}
part_c_by_query = {row['query_id']: row for row in part_c_per_query}


def make_grounded_prompt(query_row, reranked_row):
    evidence = []
    for number, item in enumerate(reranked_row['top3_after'], start=1):
        chunk = chunks_by_id[item['chunk_id']]
        evidence.append(
            f"[{number}] Source: {chunk['source']} | Chunk: {chunk['id']}\n{chunk['text']}"
        )
    evidence_text = '\n\n'.join(evidence)
    return (
        'You answer AWS questions using only the supplied evidence passages. '
        'Cite every factual claim with its passage number, such as [1]. '
        'Do not use outside knowledge or invent details. '
        'If the passages do not contain enough information, reply exactly: '
        'Not found in the documents.\n\n'
        f"Question: {query_row['prompt']}\n\n"
        f'Evidence passages:\n{evidence_text}\n\n'
        'Answer:'
    )


generation_records = []
citation_pattern = re.compile(r'\[(\d+)\]')
for query_row in queries:
    reranked_row = part_c_by_query[query_row['id']]
    prompt = make_grounded_prompt(query_row, reranked_row)
    model_inputs = mistral_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=False,
        add_special_tokens=True,
    )
    input_token_count = int(model_inputs['input_ids'].shape[1])
    if input_token_count + GENERATION_MAX_NEW_TOKENS > mistral_context_limit:
        raise ValueError(
            f"Prompt for {query_row['id']} plus the reserved answer tokens exceeds "
            f'the model context limit of {mistral_context_limit}; no truncation will be applied.'
        )

    model_inputs = {name: tensor.to(mistral_model.device) for name, tensor in model_inputs.items()}
    with torch.inference_mode():
        generated = mistral_model.generate(
            **model_inputs,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=mistral_tokenizer.pad_token_id,
            eos_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = generated[0, input_token_count:]
    answer = mistral_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    cited_numbers = sorted({int(number) for number in citation_pattern.findall(answer)})

    generation_records.append({
        'query_id': query_row['id'],
        'question': query_row['prompt'],
        'answer': answer,
        'input_token_count': input_token_count,
        'supporting_chunks': [
            {
                'number': number,
                'chunk_id': item['chunk_id'],
                'source': item['source'],
            }
            for number, item in enumerate(reranked_row['top3_after'], start=1)
        ],
        'cited_passage_numbers': cited_numbers,
        'citations_in_range': all(1 <= number <= len(reranked_row['top3_after']) for number in cited_numbers),
    })

PART_D_GENERATION_PATH = RESULTS_DIR / 'part_d_generated_answers.json'
PART_D_GENERATION_PATH.write_text(
    json.dumps(generation_records, indent=2), encoding='utf-8'
)

generation_table = [
    '| Query | Generated answer | Cited passage numbers |',
    '|---|---|---|',
]
for record in generation_records:
    safe_answer = ' '.join(record['answer'].split()).replace('|', '/')
    generation_table.append(
        f"| {record['query_id']} | {safe_answer} | {record['cited_passage_numbers']} |"
    )
display(Markdown('\n'.join(generation_table)))
print(f'Saved {len(generation_records)} grounded answers to {PART_D_GENERATION_PATH}')